In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.utils import Utility
from src.vector_store import VectorStoreManager

nist_url = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
eu_url = "https://spiderhub.cedia.edu.ec/en/documents/284/export-pdf"

nist_local_path = Utility.download_regulatory_pdf(nist_url, "NIST_AI_RMF.pdf")
eu_local_path = Utility.download_regulatory_pdf(eu_url, "EU_AI_Act.pdf")

nist_nodes = Utility.extract_metadata_chunks(nist_local_path, "NIST_AI_RMF.pdf")
eu_nodes = Utility.extract_metadata_chunks(eu_local_path, "EU_AI_Act.pdf")

all_regulatory_nodes = nist_nodes + eu_nodes
vector_db = VectorStoreManager.create_local_vector_store(all_regulatory_nodes, "data/regulatory_faiss")



In [2]:
from src.vector_store import VectorStoreManager

loaded_db = VectorStoreManager.load_local_vector_store("data/regulatory_faiss")
query = "What are the risk management requirements for AI systems?"

docs = loaded_db.similarity_search(query, k = 2)


/Users/SharminChampa/Library/CloudStorage/Dropbox/Mac (2)/Documents/LLM/Multi-Agent/src/vector_store.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9862.19it/s]


In [3]:
from src.router import GatekeeperRouter

clear_query = "What is the official risk management framework of NIST?"
route_result_1 = GatekeeperRouter.calculate_uncertainty_route(clear_query, loaded_db)

print("\n--- 🧭 ROUTER TELEMETRY: TEST 1 ---")
print(f"📊 Calculated Entropy : {route_result_1['entropy']}")
print(f"🔀 Assigned Route     : {route_result_1['selected_route']}")
print(f"🛠️ Action Triggered    : {route_result_1['action_code']}\n")

complex_query = "Compare EU Act classification with NIST and detect hidden compliance holes in high risk systems"
route_result_2 = GatekeeperRouter.calculate_uncertainty_route(complex_query, loaded_db)

print("\n--- 🧭 ROUTER TELEMETRY: TEST 2 ---")
print(f"📊 Calculated Entropy : {route_result_2['entropy']}")
print(f"🔀 Assigned Route     : {route_result_2['selected_route']}")
print(f"🛠️ Action Triggered    : {route_result_2['action_code']}")





--- 🧭 ROUTER TELEMETRY: TEST 1 ---
📊 Calculated Entropy : 0.0
🔀 Assigned Route     : Fast-Path (Direct Knowledge Retrieval)
🛠️ Action Triggered    : LOW_ENTROPY_FAST_PATH


--- 🧭 ROUTER TELEMETRY: TEST 2 ---
📊 Calculated Entropy : 1.5844999551773071
🔀 Assigned Route     : LangGraph Multi-Agent (Deep Legal Cross-Check)
🛠️ Action Triggered    : CONFLICT_DETECTED_DEEP_AUDIT


In [6]:
%autoreload 2

from src.pipeline import ComplianceEngine

pipeline = ComplianceEngine(vector_db=loaded_db)
user_query = "What is the primary objective of the NIST AI Risk Management Framework?"
result = pipeline.run(user_query)

print(f"Route Taken  : {result['route_taken']}")
print(f"Time Elapsed : {result['execution_time']} seconds\n")
print(result['output'])



Route Taken  : Fast-Path (Direct Knowledge Retrieval)
Time Elapsed : 5.37 seconds

According to the NIST AI Risk Management Framework (AI RMF) 1.0, the primary objective is "to provide a structured approach for organizations to manage and mitigate risks associated with Artificial Intelligence (AI) systems."
